# KronLLM Q10 — тест чекпоинта

Ноутбук для игры с моделью **без Kaggle**: грузит чекпоинт с диска, поднимает
`KronTransformer` из конфига, сохранённого внутри чекпоинта, и стримит генерацию
токен за токеном.

**Что за веса.** Q10 — 9,938,886 параметров, d=512, 25 слоёв, Kronecker-композиция
(terms 10, carry_window 2), 8K BPE с поразрядными цифрами. Это **8.5 % эпохи**
(шаг 11,723 из 137,377), val bpb 0.884.

**Чего ждать.** Модель держит морфологию, синтаксис и формат текста, но смысл
ещё слабый: правдоподобно продолжает фразу и теряет нить через предложение.
Арифметика на цифрах пока не работает — поразрядная токенизация даёт
возможность, а не умение; сложение появится ближе к концу эпохи. Это нормально
для 8 % обучения, а не баг.

Готовые примеры — ниже, свой промпт — в ячейке 4.

## 1. Загрузка

Чекпоинт берётся из `data/kaggle_out/q10/last.pt` (скачанный с Kaggle),
токенизатор — из `data/bilingual/tokenizer.json`. Если файла нет, ячейка
поищет другие `*.pt` в репозитории.

In [ ]:
import sys
import time
from pathlib import Path

import torch

REPO = Path(r"C:\Users\Icold\Desktop\kron_llm")
sys.path.insert(0, str(REPO))

from kronllm.model import KronTransformer
from kronllm.tokenizer import BPETokenizer

CANDIDATES = [
    REPO / "data/kaggle_out/q10/last.pt",
    REPO / "data/kaggle_ckpt/last.pt",
]
CKPT = next((p for p in CANDIDATES if p.exists()), None)
if CKPT is None:
    CKPT = max(REPO.glob("**/*.pt"), key=lambda p: p.stat().st_mtime)
TOKENIZER = REPO / "data/bilingual/tokenizer.json"

ck = torch.load(CKPT, map_location="cpu", weights_only=False)
cfg = ck["cfg"]

model = KronTransformer(cfg)
model.load_state_dict(ck["model"])
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

tok = BPETokenizer(TOKENIZER, cfg["data"]["vocab_size"], 3.694,
                   cfg["data"]["eot_id"])

params = sum(p.numel() for p in model.parameters())
print(f"checkpoint : {CKPT}")
print(f"step       : {ck.get('step')}")
print(f"val ppl    : {float(ck.get('val_ppl', float('nan'))):.3f}")
print(f"params     : {params:,}")
print(f"device     : {device}")
print(f"vocab      : {cfg['data']['vocab_size']}, eot {cfg['data']['eot_id']}")

## 2. Стриминг

`stream()` — генератор: отдаёт кусочки текста по мере появления, `show()`
печатает их сразу, без ожидания всего ответа.

**Про повторение.** У маленькой модели на раннем чекпоинте есть характерная
болезнь: она зацикливается («…KM/KM/KM», «…multi_multi»). Это не сломанный
сэмплер, а отсутствие уверенности: распределение плоское, и один раз выбранный
токен снова становится вероятным. Лечится `repetition_penalty` — мягким
штрафом для токенов, которые уже были в последних `rep_window` позициях.
Поставь `repetition_penalty=1.0`, если хочешь увидеть её «естественное»
поведение без штрафа.

In [ ]:
@torch.no_grad()
def stream(prompt, max_new_tokens=160, temperature=0.8, top_k=40,
           repetition_penalty=1.2, rep_window=64):
    # Yield generated text as it appears, one token at a time.
    ids = tok.encode(prompt)
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    past, shown = None, ""
    for _ in range(max_new_tokens):
        if past is not None and past[0][0].shape[2] < model.seq_len:
            step = model(idx[:, -1:], past_kv=past, use_cache=True)
            past = step["past_kv"]
        else:
            step = model(idx[:, -model.seq_len:], use_cache=True)
            past = step.get("past_kv")

        logits = step["logits"][:, -1, :].float()

        # Repetition penalty (CTRL-style): push down tokens already in the
        # recent window before the temperature and top-k cut.
        if repetition_penalty and repetition_penalty != 1.0:
            recent = torch.unique(idx[0, -rep_window:])
            target = logits[0, recent]
            logits[0, recent] = torch.where(
                target > 0, target / repetition_penalty, target * repetition_penalty)

        logits = logits / max(temperature, 1e-5)
        if top_k:
            k = min(top_k, logits.size(-1))
            thresh = torch.topk(logits, k, dim=-1).values[:, -1:]
            logits = logits.masked_fill(logits < thresh, float("-inf"))

        nxt = torch.multinomial(torch.softmax(logits, dim=-1), 1)
        idx = torch.cat([idx, nxt], dim=1)
        if int(nxt[0, 0]) == tok.eot_id:
            break

        # ByteLevel decode of a partial token can end mid-character; wait for
        # the next token instead of printing a replacement glyph.
        text = tok.decode(idx[0, len(ids):].tolist())
        if text.endswith("\ufffd"):
            continue
        if len(text) > len(shown):
            yield text[len(shown):]
            shown = text


def show(prompt, **kw):
    print(f"\n\u25b8 {prompt!r}\n")
    t0 = time.time()
    for piece in stream(prompt, **kw):
        print(piece, end="", flush=True)
    print(f"\n\n  [{time.time() - t0:.1f}s]")

## 3. Готовые примеры

Промпты подобраны так, чтобы было видно разные стороны: энциклопедический
регистр, английская и русская проза, код, цифры и identity-слайс.

Про identity: в корпусе есть слайс фактов о самой модели, поэтому `KronLLM is`
и `My name is` — не случайные промпты, а проба, помнит ли она себя.

In [ ]:
show("The Moon orbits the Earth at a distance of", max_new_tokens=80)
show("She opened the door and", max_new_tokens=80)
show("Он открыл дверь и", max_new_tokens=80)
show("def add(a, b):\n    return", max_new_tokens=60, temperature=0.5)
show("17 + 25 =", max_new_tokens=30, temperature=0.2, top_k=20)
show("KronLLM is", max_new_tokens=80)

## 4. Свой промпт

Меняй `PROMPT` и запускай ячейку — стриминг пойдёт сразу.

Ручки: `TEMPERATURE = 0.3` — почти детерминированно, `1.0` — свободнее;
`TOP_K = 10` — узко, `100` — широко; `REPETITION_PENALTY = 1.0` — выключить
штраф за повторы.

In [ ]:
PROMPT = "В начале двадцатого века физики обнаружили, что"
TEMPERATURE = 0.8
TOP_K = 40
MAX_NEW_TOKENS = 160
REPETITION_PENALTY = 1.2

show(PROMPT, temperature=TEMPERATURE, top_k=TOP_K,
     max_new_tokens=MAX_NEW_TOKENS, repetition_penalty=REPETITION_PENALTY)

## 5. Формат диалога (пока не инструкции)

В токенизаторе зарезервированы служебные токены под SFT:

| токен | id |
|---|---:|
| `<|system|>` | 8187 |
| `<|user|>` | 8188 |
| `<|assistant|>` | 8189 |
| `<|eot|>` | 8190 |

Сейчас модель видела их только как разделители, инструкции она не выполняет —
это задача SFT после претрейна. Ячейка показывает, как выглядит промпт в этом
формате и что модель на него отвечает сейчас.

In [ ]:
def ask(question, system="You are KronLLM, a compact bilingual model.",
        max_new_tokens=120, temperature=0.8, top_k=40,
        repetition_penalty=1.2):
    prompt = (f"<|system|>{system}<|eot|>"
              f"<|user|>{question}<|eot|><|assistant|>")
    print(f"\n\u25b8 {question}\n")
    for piece in stream(prompt, max_new_tokens=max_new_tokens,
                        temperature=temperature, top_k=top_k,
                        repetition_penalty=repetition_penalty):
        print(piece, end="", flush=True)
    print("\n")


ask("What is the Kronecker product?")

## 6. Когда появится новый чекпоинт

Скачать свежий с Kaggle:

```powershell
python -m kaggle kernels output icold21/kron-q10-10m-bilingual-pretraining -p data\kaggle_out
```

Потом достаточно перезапустить ячейки 1–2: веса и токенизатор подхватятся сами,
а `step` и `val ppl` наверху покажут, что именно загружено.

После decay-прохода (последние 20 % эпохи, LR 3e-4 → 1e-6) чекпоинт будет
заметно лучше на фактах и identity — тогда же имеет смысл сравнивать
`epoch1.pt` против `annealed.pt` на одних и тех же промптах.